# Demo — Lasso 10M FineWeb documents, read what is in your cluster

**The canonical ML researcher workflow.** Load a real embedding, find a
visual cluster, lasso it, get back the row indices, read the actual text of
what was in that cluster.

Data: 10,000,000 documents from a FineWeb corpus, with 2D embedding
positions and a parallel text blob keyed by row offset. Files live in
the research repo at `Projects/Research/Stipple/app/public/data/`.


In [1]:
import mmap
import numpy as np
from pathlib import Path
from stipple import Stipple

DATA = Path.home() / 'Sites/Thinking/Projects/Research/Stipple/app/public/data'

# 10M rows × 10 f32 columns. Cols 0,1 = embedding x,y; col 2 = jaccard.
arr = np.fromfile(DATA / 'fineweb-10m-gpu-chunk00.f32', dtype=np.float32).reshape(-1, 10)
xs = np.ascontiguousarray(arr[:, 0])
ys = np.ascontiguousarray(arr[:, 1])

# Text payload: byte offsets index into a 7.9 GB blob. mmap so only pages
# we read actually hit memory.
offsets = np.fromfile(DATA / 'fineweb-10m-gpu-chunk00-text-offsets.bin', dtype=np.uint64)
_blob_f = open(DATA / 'fineweb-10m-gpu-chunk00-text-blob.bin', 'rb')
text_blob = mmap.mmap(_blob_f.fileno(), 0, access=mmap.ACCESS_READ)

def doc(i):
    start = int(offsets[i])
    end = int(offsets[i+1]) if i+1 < len(offsets) else start + 2000
    return text_blob[start:end].decode('utf-8', errors='replace')

print(f'{len(xs):,} FineWeb documents · text blob {text_blob.size() / (1<<30):.1f} GB')

10,000,000 FineWeb documents · text blob 7.3 GB


## Render

`render_mode='density'` because per-point scatter at 10M would be slow.
Density-mode draws a 1024² bin grid heatmap — work scales with bins, not
rows. Lasso is still available because positions are retained on GPU.

**Try it:** pan with click-drag, zoom with the scroll wheel, then
**shift+drag** around a cluster you want to inspect.

In [2]:
w = Stipple(x=xs, y=ys, render_mode='density')
w

## Inspect the lassoed cluster

Re-run this cell after lassoing a region. Stipple syncs `selected_indices`
back to Python; we then read the corresponding documents straight from the
text blob.

In [4]:
idx = w.selected_indices
if len(idx) == 0:
    print('No selection yet. Shift+drag in the widget above to lasso a region, then re-run this cell.')
else:
    print(f'Lassoed {len(idx):,} documents · gpu round-trip {w.selection_ms:.1f} ms')
    print()
    # Random sample of 5 to avoid biasing toward early rows.
    sample = np.random.default_rng(0).choice(idx, size=min(5, len(idx)), replace=False)
    for row in sorted(sample.tolist()):
        text = doc(int(row))[:240].replace(chr(10), ' ').strip()
        print(f'--- row {row:,} ---')
        print(text)
        print()

No selection yet. Shift+drag in the widget above to lasso a region, then re-run this cell.
